In [1]:
import os
import time
import sys
import clr
import numpy as np
import matplotlib.pyplot as plt
from slmcontrol import *
import time 
from pylablib.devices import Thorlabs
from slmcontrol import *
from cameras.Ximea import XimeaCamera
from cameras.ImagingSource import ImagingSourceCamera

# Add References to .NET libraries

clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.DeviceManagerCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.GenericMotorCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\ThorLabs.MotionControl.KCube.StepperMotorCLI.dll")
from Thorlabs.MotionControl.DeviceManagerCLI import *
from Thorlabs.MotionControl.GenericMotorCLI import *
from Thorlabs.MotionControl.KCube.StepperMotorCLI import *
from System import Decimal  # necessary for real world units

#input beam



#PARA O EXPERIMENTO RODA ESSE

In [ ]:
print(np.linspace(0.3, 0.6, 7))

In [2]:
slm = SLMdisplay(monitor=1)
config_path = 'config.ini'
config = configparser.ConfigParser()
config.read(config_path)
x,y = build_grid(config_path)
input = hg(x,y,0,0,config['incoming'].getfloat('waist'))



In [4]:
desired = hg(x, y, 1 , 0, 0.5)
holo = generate_hologram(desired,input,x,y,195,3,4,0,0,method='bessel1')
slm.updateArray(holo)

In [ ]:
def move_to_position(serial_no, new_position):
    """Function to move the motor to a new position"""

    try:
        DeviceManagerCLI.BuildDeviceList()

        # Create new device
        device = KCubeStepper.CreateKCubeStepper(serial_no)

        # Connect
        device.Connect(serial_no)
        time.sleep(0.25)  # Wait statements are important to allow settings to be sent to the device

        # Start polling and enable
        device.StartPolling(250)  # 250ms polling rate
        time.sleep(0.25)
        device.EnableDevice()
        time.sleep(0.25)  # Wait for device to enable

        # Load device configuration
        use_file_settings = DeviceConfiguration.DeviceSettingsUseOptionType.UseFileSettings
        device_config = device.LoadMotorConfiguration(device.DeviceID, use_file_settings)

        # Set homing parameters to position 10 mm
        home_params = device.GetHomingParams()
        home_params.Position = Decimal(10.0)  # Set home position to 10 mm
        device.SetHomingParams(home_params)


        # Set velocity parameters for slower movement
        vel_params = device.GetVelocityParams()
        vel_params.MaxVelocity = Decimal(1.0)  # Set desired max velocity in real units
        vel_params.Acceleration = Decimal(0.5)  # Set desired acceleration in real units
        device.SetVelocityParams(vel_params)

        # Move to new position
        new_pos = Decimal(new_position)  # in Real Units
        print(f"Moving to position {new_pos}...")
        device.MoveTo(new_pos, 60000)  # 60 seconds timeout
        print(f"Reached position {new_pos}.")

        # Stop Polling and Disconnect
        device.StopPolling()
        device.Disconnect()
    except Exception as e:
        print(e)

def main():
    """The main entry point for the application"""
    serial_no = "26005619"  # Replace this line with your device's serial number
    
    for i in np.linspace(0,10):

        if i == 0:
            move_to_position(serial_no, 10)
            time.sleep(1)
            print('home')
        step = i + 10 
        time.sleep(0.5)
        move_to_position(serial_no, step)
        time.sleep(0.5)
        desired = lg(x, y, 0, i, .5)
        holo = generate_hologram(desired,input,x,y,195,3,4,0,0,method='bessel1')
        slm.updateArray(holo)
        time.sleep(0.5)
        img = camera.capture()
        np.savez(file_path, imgs = img, infos = infos)


if __name__ == "__main__":
    main()

In [ ]:

camera = ImagingSourceCamera()



#input beam
slm = SLMdisplay(monitor=1)
config_path = 'config.ini'
config = configparser.ConfigParser()
config.read(config_path)
x,y = build_grid(config_path)
input = hg(x,y,0,0,config['incoming'].getfloat('waist'))

stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
print(f" stage z  -- {stage_z.get_position(scale= 'ZST')}")
stages = [stage_z]

for _ in stages:
    print(_.is_homed())
    if _.is_homed() == False:
        _.home()
for _ in stages:
    _.wait_for_home()

stage_z.move_to(0.004) # in meter
stage_z.wait_move()
resolution = 100
step_size = 100e-9

ls = 1                  #laguerre vai até quanto?


#data = np.empty((resolution,ls,480, 640),dtype="uint8")
data = np.empty((resolution,480, 640),dtype="uint8")

"""for i in range(resolution):
        stage_z.move_by(step_size)
        stage_z.wait_move()
        for n in range(ls):
            desired = hg(x, y, n +1, 0, .5)
            holo = generate_hologram(desired,input,x,y,195,3,4,0,0,method='bessel1')
            slm.updateArray(holo)
            time.sleep(0.15)
            data[i,n] = camera.capture()"""

desired = hg(x, y, 2, 1, .5)
holo = generate_hologram(desired,input,x,y,195,3,4,0,0,method='bessel1')
slm.updateArray(holo)

for i in range(resolution):
    stage_z.move_by(step_size)
    stage_z.wait_move()
    time.sleep(0.15)
    data[i] = camera.capture()

camera.close()


#E ESSE

In [ ]:
camera.capture()

In [ ]:
camera.close()

In [ ]:
camera = ImagingSourceCamera()

p = camera.capture()
cut = p < 253
p = p*cut
camera.close()
plt.imshow(p)
plt.colorbar()

In [ ]:
stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
print(f" stage z  -- {stage_z.get_position(scale= 'ZST')}")
stages = [stage_z]


In [ ]:

stage_z.move_to(0.004) # in meter
stage_z.wait_move()

In [ ]:
#plt.imshow(data[1,2])
plt.imshow(data[1])
plt.colorbar()

#ESSE PARA VER UMA FOTO

In [ ]:
np.savez('hg21 step100e-9', data)

#ESSE PARA SALVAR, ESCOLHE UM NOME AI
#FIM

In [ ]:
data = np.load('hg30 step100e-9.npz')['arr_0']
cut = data < 250
data = data*cut
plt.imshow(data[15])
plt.colorbar()


In [ ]:
data = np.load('data_test2.npz')['arr_0']
cut = data < 250
data = data*cut
plt.imshow(data[0,0])
plt.colorbar()

In [ ]:
t0 = time.time()
for _ in range(100):
    stage_z.move_by(80e-9)
    stage_z.wait_move()
t1 = time.time()
(t1 - t0) / 100

In [ ]:
stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
stage_z.move_to(.01)

In [ ]:
import numpy as np
x = np.linspace(-10,10,21)
print(x)